# 00 — Regional flows at the 100-mile cutoff (prerequisite + validation gate)

Re-runs the regional SARIMAX baselines **and** disruption/recovery metrics at the **100-mile** affected
region, for **within / inflow / outflow x 6+1 categories x 2 storms**. Feeds notebooks 01 (1c worked
example), 02 (category heatmaps), 03 (category recovery).

**Reuses** `notebook/regional_baseline.ipynb` + `notebook/regional_metrics.ipynb` **verbatim**, with only:
1. county list -> `results/local_level/{hrc}_100mi/counties_geoid_cut_100.txt` (Helene 487, Milton 34),
2. `cutoff_mile = 100`,
3. relative paths adjusted for this notebook's deeper location (`npj/notebook/`).

Utilities is still computed (completeness) but **excluded from npj figures** (`CATEGORIES_NPJ` = 6).

**Outputs** -> `results/npj_100mi/regional_data/{hrc}/baseline_{flow}_{cat}.csv`
and `results/npj_100mi/regional_data/regional_metrics_summary_100mi.csv`.

The final cell is the **validation gate**: Spearman rank correlation of per-category drop/recovery
between the 50-mi and 100-mi cutoffs (confirms category rankings are stable -> 100-mi-everywhere is safe).

In [1]:
import pandas as pd
import numpy as np
import h5py
import os
import sys
import datetime as dt
from importlib import reload

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import theilslopes, spearmanr

# This notebook lives in npj/notebook/ -> one level deeper than the original notebook/.
HURRICANE_OCT = './../../../hurricane_oct/'        # sibling repo: mobility H5, mobility_function
NOTEBOOK_DIR  = '../../notebook'                    # original notebook dir: geoid map + recovery fns
RESULTS       = '../../results'
OUTPUT_ROOT   = f'{RESULTS}/npj_100mi/regional_data'

sys.path.append(HURRICANE_OCT)
sys.path.append(os.path.join(HURRICANE_OCT, 'mobility_function'))
sys.path.insert(0, NOTEBOOK_DIR)
from mobility_function import analysis as ma
ma = reload(ma)
from recovery_function_v2 import (prepare_time_series_with_exog,
                                  fit_arimax_model,
                                  get_predictions_and_ci)

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('Setup complete. Output ->', OUTPUT_ROOT)

Setup complete. Output -> ../../results/npj_100mi/regional_data


In [2]:
# Date helpers (identical to notebook/regional_baseline.ipynb)
def mondays_str(year, start_month=7, end_month=10):
    start = dt.date(year, start_month, 28)
    end = dt.date(year, end_month, 31)
    days_ahead = (0 - start.weekday()) % 7
    cur = start + dt.timedelta(days=days_ahead)
    out = []
    while cur <= end:
        out.append(cur.strftime('%Y%m%d'))
        cur += dt.timedelta(days=7)
    return out

mondays_2023 = mondays_str(2023)
mondays_2024 = mondays_str(2024)
all_mondays = mondays_2023 + mondays_2024

dates_2023 = pd.date_range(start='2023-07-31', periods=len(mondays_2023)*7, freq='D')
dates_2024 = pd.date_range(start='2024-07-29', periods=len(mondays_2024)*7, freq='D')
dates_all = dates_2023.union(dates_2024)

# 17 raw POI categories -> 7 groups. Utilities COMPUTED for completeness but EXCLUDED from npj figures.
GROUPS = {
    'Travel':              [1, 2, 3, 4, 5, 12],
    'Work & Professional': [9],
    'Health':              [14],
    'Education':           [13],
    'Retail & Leisure':    [11, 15],
    'Urban Government':    [6, 7, 8, 16],
    'Utilities':           [0, 10],
}
group_names = list(GROUPS.keys())
CATEGORIES_NPJ = [g for g in group_names if g != 'Utilities']   # the 6 figure categories

ARIMA_ORDER = (1, 0, 0)
SEASONAL_ORDER = (0, 0, 0, 0)
print(f'Mondays {len(all_mondays)} | Days {len(dates_all)} | Categories {len(group_names)} (npj uses {len(CATEGORIES_NPJ)})')

Mondays 28 | Days 196 | Categories 7 (npj uses 6)


In [3]:
# Hurricane configs — 100-mi cutoff
HURRICANE_CONFIGS = {
    'milton': {'landing_date': pd.Timestamp('2024-10-09'), 'cutoff_mile': 100,
               'county_list': f'{RESULTS}/local_level/milton_100mi/counties_geoid_cut_100.txt'},
    'helene': {'landing_date': pd.Timestamp('2024-09-26'), 'cutoff_mile': 100,
               'county_list': f'{RESULTS}/local_level/helene_100mi/counties_geoid_cut_100.txt'},
}
FLOW_TYPES = ['within', 'inflow', 'outflow']

In [4]:
# Load regional flows for the 100-mi region (identical logic to source; only county_list path changed)
def load_regional_flows(hrc_name, county_list_path):
    county_list = [int(line.strip()) for line in open(county_list_path)]
    geo_idx = pd.read_csv(f'{NOTEBOOK_DIR}/geoid_idx_names.csv')
    geo_idx['GEOID'] = geo_idx['GEOID'].astype(int)
    selected_idx = geo_idx[geo_idx['GEOID'].isin(county_list)].county_idx.values
    print(f'  {hrc_name}: {len(selected_idx)} counties (of {len(county_list)})')

    W_all, O_all, I_all = [], [], []
    for date_str in all_mondays:
        M = ma.h5py_to_4d_array(HURRICANE_OCT + f'data/mobility/M_raw_{date_str}.h5')
        M_w, M_o, M_i = ma.region_mobility(M, selected_idx)
        W_all.append(M_w.sum(axis=2)); O_all.append(M_o.sum(axis=2)); I_all.append(M_i.sum(axis=2))
    W_ts = np.concatenate(W_all, axis=0); O_ts = np.concatenate(O_all, axis=0); I_ts = np.concatenate(I_all, axis=0)

    n_days, n_groups = W_ts.shape[0], len(group_names)
    def merge_cats(raw_ts):
        merged = np.zeros((n_days, n_groups))
        for g, name in enumerate(group_names):
            merged[:, g] = raw_ts[:, GROUPS[name]].sum(axis=1)
        return merged
    return {'within': merge_cats(W_ts), 'outflow': merge_cats(O_ts), 'inflow': merge_cats(I_ts)}

all_flow_data = {}
for hrc_name, cfg in HURRICANE_CONFIGS.items():
    print(f'Loading {hrc_name} (100 mi)...')
    all_flow_data[hrc_name] = load_regional_flows(hrc_name, cfg['county_list'])
    print(f'  shape {all_flow_data[hrc_name]["within"].shape}')

Loading milton (100 mi)...
  milton: 34 counties (of 34)


  shape (196, 7)
Loading helene (100 mi)...
  helene: 487 counties (of 487)


  shape (196, 7)


## Fit SARIMAX baselines (100-mi) and export

In [5]:
for hrc_name, hrc_cfg in HURRICANE_CONFIGS.items():
    landing_date = hrc_cfg['landing_date']
    train_end = (landing_date - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    forecast_start = (landing_date - pd.Timedelta(days=6)).strftime('%Y-%m-%d')
    forecast_end = '2024-10-31'
    out_dir = f'{OUTPUT_ROOT}/{hrc_name}'; fig_dir = f'{out_dir}/figures'
    os.makedirs(fig_dir, exist_ok=True)
    flow_data = all_flow_data[hrc_name]
    print(f"\n{'='*60}\n{hrc_name.upper()} (landing {landing_date.date()})\n{'='*60}")
    for ft_name in FLOW_TYPES:
        ts_array = flow_data[ft_name]
        for g, cat_name in enumerate(group_names):
            safe_cat = cat_name.replace(' ', '_').replace('&', 'and')
            csv_path = f'{out_dir}/baseline_{ft_name}_{safe_cat}.csv'
            fig_path = f'{fig_dir}/baseline_{ft_name}_{safe_cat}.png'
            flow_y = ts_array[:, g]
            try:
                y_log, y, X = prepare_time_series_with_exog(flow_y, dates_all)
                res, y_train, X_train = fit_arimax_model(
                    y_log, X, order=ARIMA_ORDER, seasonal_order=SEASONAL_ORDER, train_2024_end=train_end)
                df_rec, forecast_idx = get_predictions_and_ci(
                    res, X, y, forecast_start=forecast_start, forecast_end=forecast_end)
                df_rec.to_csv(csv_path)
                fig, ax = plt.subplots(figsize=(14, 5))
                ax.plot(df_rec.index, df_rec['y_true'], 'k-', lw=1.5, label='Observed')
                ax.plot(df_rec.index, df_rec['y_pred'], 'r-', lw=1.5, label='Predicted (baseline)')
                ax.fill_between(df_rec.index, df_rec['ci_lower'], df_rec['ci_upper'], color='red', alpha=0.15)
                ax.axvline(landing_date, color='blue', ls='--', lw=2)
                ax.set_title(f'{hrc_name.capitalize()} - {cat_name} - {ft_name} (100 mi)', fontweight='bold')
                ax.legend(fontsize=9); ax.grid(True, alpha=0.3); plt.xticks(rotation=45); plt.tight_layout()
                plt.savefig(fig_path, dpi=150, bbox_inches='tight'); plt.close()
            except Exception as e:
                print(f'  [{ft_name}] {cat_name}: FAILED - {e}')
    n = len([f for f in os.listdir(out_dir) if f.startswith('baseline_') and f.endswith('.csv')])
    print(f'  baselines saved: {n} CSVs')


MILTON (landing 2024-10-09)


  baselines saved: 21 CSVs

HELENE (landing 2024-09-26)


/Users/qing/miniconda3/envs/geo/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


/Users/qing/miniconda3/envs/geo/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  baselines saved: 21 CSVs


## Disruption + recovery metrics (functions verbatim from notebook/regional_metrics.ipynb)

In [6]:
def compute_relative_deviation(df):
    eps = 1e-12
    denom = df['y_pred'].replace(0, np.nan) + eps
    rel_diff = (df['y_true'] - df['y_pred']) / denom * 100
    rel_lower = (df['ci_lower'] - df['y_pred']) / denom * 100
    rel_upper = (df['ci_upper'] - df['y_pred']) / denom * 100
    return rel_diff, rel_lower, rel_upper

def compute_largest_drop(rel_diff, landing_date, window_days=6):
    start = landing_date; end = landing_date + pd.Timedelta(days=window_days)
    window = rel_diff.loc[(rel_diff.index >= start) & (rel_diff.index <= end)]
    if window.empty: return None, None
    return float(window.min()), window.idxmin()

def compute_outflow_increase(rel_diff, landing_date, pre_days=3, post_days=6):
    start = landing_date - pd.Timedelta(days=pre_days); end = landing_date + pd.Timedelta(days=post_days)
    window = rel_diff.loc[(rel_diff.index >= start) & (rel_diff.index <= end)]
    if window.empty: return None, None
    return float(window.max()), window.idxmax()

def trend_based_recovery(rel_diff, landing_date, smooth_window=3, trough_search_days=10):
    rd_smooth = rel_diff.rolling(window=smooth_window, center=True, min_periods=1).mean()
    trough_end = landing_date + pd.Timedelta(days=trough_search_days)
    search = rd_smooth.loc[(rd_smooth.index >= landing_date) & (rd_smooth.index <= trough_end)]
    if search.empty or search.min() >= 0:
        return {'trough_date': None, 'recovery_days': None, 'slope': None, 'rd_smooth': rd_smooth}
    trough_date = search.idxmin()
    post_trough = rd_smooth.loc[rd_smooth.index >= trough_date]; vals = post_trough.values
    mono_end = 1
    for i in range(1, len(vals)):
        if vals[i] >= vals[i-1] - 1e-15: mono_end = i + 1
        else: break
    mono_segment = post_trough.iloc[:mono_end]
    if len(mono_segment) < 2:
        return {'trough_date': trough_date, 'recovery_days': None, 'slope': None, 'rd_smooth': rd_smooth}
    t = np.arange(len(mono_segment), dtype=float)
    slope, intercept, lo_slope, hi_slope = theilslopes(mono_segment.values, t)
    if slope <= 0:
        return {'trough_date': trough_date, 'recovery_days': None, 'slope': float(slope), 'rd_smooth': rd_smooth}
    tau_from_trough = -intercept / slope
    recovery_days = (trough_date - landing_date).days + tau_from_trough
    return {'trough_date': trough_date, 'recovery_days': float(recovery_days),
            'slope': float(slope), 'intercept': float(intercept), 'rd_smooth': rd_smooth}

print('Metric functions defined.')

Metric functions defined.


In [7]:
# Compute metrics from the saved 100-mi baselines -> summary CSV
GROUP_DISPLAY = {'Travel':'Travel','Work_and_Professional':'Work & Professional','Health':'Health',
                 'Education':'Education','Retail_and_Leisure':'Retail & Leisure',
                 'Urban_Government':'Urban Government','Utilities':'Utilities'}
SAFE = list(GROUP_DISPLAY.keys())

rows = []
for hrc_name, cfg in HURRICANE_CONFIGS.items():
    landing = cfg['landing_date']; out_dir = f'{OUTPUT_ROOT}/{hrc_name}'
    for ft in FLOW_TYPES:
        for safe in SAFE:
            p = f'{out_dir}/baseline_{ft}_{safe}.csv'
            if not os.path.exists(p): continue
            df = pd.read_csv(p, index_col=0, parse_dates=True)
            rel, _, _ = compute_relative_deviation(df)
            row = {'hurricane': hrc_name, 'flow_type': ft, 'category': GROUP_DISPLAY[safe]}
            dv, dd = compute_largest_drop(rel, landing);  row['largest_drop'] = dv
            iv, idd = compute_outflow_increase(rel, landing); row['largest_increase'] = iv
            if ft in ('within', 'inflow'):
                rec = trend_based_recovery(rel, landing)
                row['recovery_days'] = rec.get('recovery_days')
                row['slope_pct_per_day'] = (rec['slope'] * 100) if rec.get('slope') else None
            else:
                row['recovery_days'] = None; row['slope_pct_per_day'] = None
            rows.append(row)

metrics_df = pd.DataFrame(rows)
summary_path = f'{OUTPUT_ROOT}/regional_metrics_summary_100mi.csv'
metrics_df.to_csv(summary_path, index=False)
print('saved', summary_path, '|', len(metrics_df), 'rows')
metrics_df[metrics_df.flow_type == 'within'][['hurricane','category','largest_drop','recovery_days','largest_increase']].round(2)

saved ../../results/npj_100mi/regional_data/regional_metrics_summary_100mi.csv | 42 rows


,hurricane,category,largest_drop,recovery_days,largest_increase
0,milton,Travel,-44.69,4.75,17.58
1,milton,Work & Professional,-29.94,4.91,16.29
2,milton,Health,-24.31,4.75,20.09
3,milton,Education,-33.62,6.05,10.23
4,milton,Retail & Leisure,-28.66,5.20,13.52
5,milton,Urban Government,-26.69,5.01,14.87
6,milton,Utilities,-39.06,4.17,21.68
21,helene,Travel,-10.94,5.09,11.44
22,helene,Work & Professional,-11.96,4.61,7.52
23,helene,Health,-11.68,4.49,8.79


## Validation gate — are category patterns stable between 50-mi and 100-mi?

If per-category drop/recovery rankings track across cutoffs (high Spearman rho), the regional category
story is robust and **100-mi everywhere** is justified.

In [8]:
old = pd.read_csv(f'{RESULTS}/regional_level/regional_metrics_summary.csv')   # existing 50-mi
new = metrics_df.copy()
NPJ6 = set(CATEGORIES_NPJ)

def cmp_table(flow, dv):
    o = old[old.flow_type == flow].set_index(['hurricane','category'])[dv].rename('50mi')
    n = new[new.flow_type == flow].set_index(['hurricane','category'])[dv].rename('100mi')
    j = pd.concat([o, n], axis=1)
    return j[j.index.get_level_values('category').isin(NPJ6)].round(2)

for flow, dv in [('within','largest_drop'), ('within','recovery_days'),
                 ('inflow','largest_drop'), ('inflow','recovery_days')]:
    t = cmp_table(flow, dv).dropna()
    rho = spearmanr(t['50mi'], t['100mi']).correlation if len(t) > 2 else float('nan')
    print(f"\n[{flow} / {dv}]  Spearman(50mi,100mi) = {rho:.3f}")
    print(t.to_string())


[within / largest_drop]  Spearman(50mi,100mi) = 0.979
                                50mi  100mi
hurricane category                         
milton    Travel              -48.62 -44.69
          Work & Professional -37.23 -29.94
          Health              -25.11 -24.31
          Education           -34.95 -33.62
          Retail & Leisure    -33.09 -28.66
          Urban Government    -29.38 -26.69
helene    Travel              -12.15 -10.94
          Work & Professional -12.58 -11.96
          Health              -10.04 -11.68
          Education            -8.46  -7.79
          Retail & Leisure    -14.61 -12.19
          Urban Government    -12.66 -13.25

[within / recovery_days]  Spearman(50mi,100mi) = 0.804
                               50mi  100mi
hurricane category                        
milton    Travel               4.83   4.75
          Work & Professional  4.79   4.91
          Health               4.55   4.75
          Education            6.31   6.05
          Retai